# ChatGPT Archive Compiler — first functional Colab run

This notebook creates a durable Git checkout in the attached **ChatGPT Data Export** Google Drive folder, installs the package from that checkout, and validates the complete ZIP → Archive IR vertical slice with synthetic data.

Privacy boundary: Colab executes on Google-hosted infrastructure. The synthetic validation is safe by design. Real ChatGPT exports may contain deeply sensitive messages, files, identifiers, and credentials; real-data execution is therefore isolated in a final cell that is disabled by default and requires explicit acknowledgment. No cell prints message content or conversation titles.

Before running, create a Colab secret named `GITHUB_TOKEN` containing a fine-grained token with **read-only Contents access** to the private `jcollins-bioinfo/chatgpt-archive-compiler` repository. Grant this notebook access to the secret. The token is passed through a temporary `GIT_ASKPASS` environment and is never placed in a URL, command argument, notebook output, or Git configuration.


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

In [ ]:
from pathlib import Path

REPO_BRANCH = "feature/00-colab-bootstrap"  # @param {type:"string"}
REPO_FULL_NAME = "jcollins-bioinfo/chatgpt-archive-compiler"
PUBLIC_REPO_URL = f"https://github.com/{REPO_FULL_NAME}.git"
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/ChatGPT Data Export")
REPO_DIR = DRIVE_PROJECT_DIR / "chatgpt-archive-compiler"
OUTPUT_ROOT = DRIVE_PROJECT_DIR / "outputs"

if not DRIVE_PROJECT_DIR.is_dir():
    raise RuntimeError("Expected Google Drive project folder is missing: ChatGPT Data Export")
if not REPO_BRANCH.strip():
    raise ValueError("REPO_BRANCH must not be empty")

print(f"Drive project directory: {DRIVE_PROJECT_DIR}")
print(f"Repository branch: {REPO_BRANCH}")

In [ ]:
import os
import subprocess
import sys
import tempfile
from collections.abc import Iterator, Mapping, Sequence
from contextlib import contextmanager
from urllib.parse import urlsplit


def run_command(
    command: Sequence[str],
    *,
    cwd: Path | None = None,
    env: Mapping[str, str] | None = None,
    capture_output: bool = False,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    """Run a shell-free subprocess with explicit arguments."""

    return subprocess.run(
        list(command),
        cwd=cwd,
        env=dict(env) if env is not None else None,
        check=check,
        text=True,
        capture_output=capture_output,
    )


def get_github_token() -> str:
    """Read the private-repository token without exposing retrieval errors."""

    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError(
            "Colab secret GITHUB_TOKEN is unavailable or notebook access is disabled."
        ) from None
    if not token:
        raise RuntimeError("Colab secret GITHUB_TOKEN is empty.")
    return token


ASKPASS_SOURCE = """#!/usr/bin/env python3
import os
import sys

prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ""
print("x-access-token" if "username" in prompt else os.environ["CAC_GIT_TOKEN"])
"""


@contextmanager
def authenticated_git_environment() -> Iterator[dict[str, str]]:
    """Yield a subprocess environment using a temporary token-free askpass file."""

    token = get_github_token()
    with tempfile.TemporaryDirectory(prefix="cac_git_auth_") as temporary_directory:
        helper = Path(temporary_directory) / "askpass.py"
        helper.write_text(ASKPASS_SOURCE, encoding="utf-8")
        helper.chmod(0o700)
        environment = os.environ.copy()
        environment.pop("GITHUB_TOKEN", None)
        environment.update(
            {
                "CAC_GIT_TOKEN": token,
                "GIT_ASKPASS": str(helper),
                "GIT_TERMINAL_PROMPT": "0",
            }
        )
        yield environment


def origin_targets_expected_repository(origin: str) -> bool:
    """Validate a GitHub origin without printing possibly credentialized input."""

    if origin.startswith("git@github.com:"):
        repository_path = origin.split(":", 1)[1]
    else:
        parsed = urlsplit(origin)
        if parsed.hostname != "github.com":
            return False
        repository_path = parsed.path
    return repository_path.strip("/").removesuffix(".git") == REPO_FULL_NAME

In [ ]:
def clone_or_update_repository() -> str:
    """Create or fast-forward a clean, authenticated Drive-backed checkout."""

    valid_branch = (
        run_command(
            ["git", "check-ref-format", "--branch", REPO_BRANCH],
            capture_output=True,
            check=False,
        ).returncode
        == 0
    )
    if not valid_branch:
        raise ValueError("REPO_BRANCH is not a valid Git branch name.")
    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(
                "Repository target exists but is not a Git checkout; refusing to overwrite it."
            )
        origin = run_command(
            ["git", "remote", "get-url", "origin"],
            cwd=REPO_DIR,
            capture_output=True,
        ).stdout.strip()
        if not origin_targets_expected_repository(origin):
            raise RuntimeError("Existing checkout origin is not the expected repository.")
        dirty = run_command(
            ["git", "status", "--porcelain"],
            cwd=REPO_DIR,
            capture_output=True,
        ).stdout
        if dirty:
            raise RuntimeError("Existing checkout has local changes; refusing to update it.")
        run_command(["git", "remote", "set-url", "origin", PUBLIC_REPO_URL], cwd=REPO_DIR)
        with authenticated_git_environment() as environment:
            run_command(
                [
                    "git",
                    "-c",
                    "credential.helper=",
                    "fetch",
                    "origin",
                    f"+refs/heads/{REPO_BRANCH}:refs/remotes/origin/{REPO_BRANCH}",
                ],
                cwd=REPO_DIR,
                env=environment,
            )
        branch_exists = (
            run_command(
                ["git", "show-ref", "--verify", f"refs/heads/{REPO_BRANCH}"],
                cwd=REPO_DIR,
                capture_output=True,
                check=False,
            ).returncode
            == 0
        )
        if branch_exists:
            run_command(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR)
            run_command(["git", "merge", "--ff-only", f"origin/{REPO_BRANCH}"], cwd=REPO_DIR)
        else:
            run_command(
                ["git", "checkout", "-b", REPO_BRANCH, "--track", f"origin/{REPO_BRANCH}"],
                cwd=REPO_DIR,
            )
    else:
        with authenticated_git_environment() as environment:
            run_command(
                [
                    "git",
                    "-c",
                    "credential.helper=",
                    "clone",
                    "--branch",
                    REPO_BRANCH,
                    "--single-branch",
                    PUBLIC_REPO_URL,
                    str(REPO_DIR),
                ],
                env=environment,
            )
    run_command(["git", "config", "--local", "core.fileMode", "false"], cwd=REPO_DIR)
    return run_command(
        ["git", "rev-parse", "--short=12", "HEAD"],
        cwd=REPO_DIR,
        capture_output=True,
    ).stdout.strip()


commit_sha = clone_or_update_repository()
print(f"Repository ready: {REPO_DIR}")
print(f"Checked-out commit: {commit_sha}")

In [ ]:
import importlib

run_command([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[notebooks]"])
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "chatgpt_archive_compiler" or module_name.startswith(
        "chatgpt_archive_compiler."
    ):
        del sys.modules[module_name]

archive_compiler = importlib.import_module("chatgpt_archive_compiler")
ingest_module = importlib.import_module("chatgpt_archive_compiler.ingest")
serialization_module = importlib.import_module("chatgpt_archive_compiler.serialization")
IngestLimits = ingest_module.IngestLimits
SchemaMode = ingest_module.SchemaMode
ingest_export_zip = ingest_module.ingest_export_zip
summarize_archive = ingest_module.summarize_archive
read_archive_ir = serialization_module.read_archive_ir
write_archive_ir = serialization_module.write_archive_ir

imported_from = Path(archive_compiler.__file__).resolve()
if REPO_DIR.resolve() not in imported_from.parents:
    raise RuntimeError("Package import did not resolve to the Drive-backed checkout.")
print(f"Installed package version: {archive_compiler.__version__}")
print(f"Imported from: {imported_from}")

## Synthetic end-to-end validation

The fixture contains a structural root, one user message, the selected assistant response, and a sibling regenerated assistant response. The assertions prove that the Archive IR preserves the complete graph while deriving only the declared current path. The synthetic ZIP remains in temporary runtime storage; only the validated IR is written to Drive.


In [ ]:
import json
import zipfile

synthetic_conversation = {
    "id": "synthetic-conversation-001",
    "title": "Synthetic branched conversation",
    "create_time": 1_735_689_600,
    "update_time": 1_735_689_700,
    "current_node": "assistant-current",
    "mapping": {
        "root": {
            "id": "root",
            "parent": None,
            "children": ["user-001"],
            "message": None,
        },
        "user-001": {
            "id": "user-001",
            "parent": "root",
            "children": ["assistant-current", "assistant-alternate"],
            "message": {
                "id": "message-user-001",
                "author": {"role": "user"},
                "create_time": 1_735_689_600,
                "content": {"content_type": "text", "parts": ["Synthetic question"]},
                "metadata": {},
            },
        },
        "assistant-current": {
            "id": "assistant-current",
            "parent": "user-001",
            "children": [],
            "message": {
                "id": "message-assistant-current",
                "author": {"role": "assistant"},
                "create_time": 1_735_689_700,
                "content": {"content_type": "text", "parts": ["Current answer"]},
                "metadata": {"model_slug": "synthetic-model"},
            },
        },
        "assistant-alternate": {
            "id": "assistant-alternate",
            "parent": "user-001",
            "children": [],
            "message": {
                "id": "message-assistant-alternate",
                "author": {"role": "assistant"},
                "create_time": 1_735_689_650,
                "content": {"content_type": "text", "parts": ["Alternate answer"]},
                "metadata": {},
            },
        },
    },
}

with tempfile.TemporaryDirectory(prefix="cac_synthetic_") as temporary_directory:
    fixture_zip = Path(temporary_directory) / "synthetic-export.zip"
    with zipfile.ZipFile(fixture_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive_zip:
        archive_zip.writestr(
            "conversations.json", json.dumps([synthetic_conversation], allow_nan=False)
        )
    archive = ingest_export_zip(fixture_zip, limits=IngestLimits(), schema_mode=SchemaMode.TOLERANT)

summary = summarize_archive(archive)
conversation = archive.conversations[0]
nodes_by_id = {node.node_id: node for node in conversation.nodes}
assert archive.archive_version == "1.0"
assert summary.conversation_count == 1
assert summary.node_count == 4
assert summary.message_count == 3
assert summary.current_path_message_count == 2
assert summary.warning_count == 0
assert set(nodes_by_id) == {"root", "user-001", "assistant-current", "assistant-alternate"}
assert conversation.current_path_node_ids == ["root", "user-001", "assistant-current"]
assert all(
    nodes_by_id[node_id].is_on_current_path for node_id in conversation.current_path_node_ids
)
assert not nodes_by_id["assistant-alternate"].is_on_current_path
assert nodes_by_id["assistant-current"].message.message_id == "message-assistant-current"
assert nodes_by_id["assistant-alternate"].message.message_id == "message-assistant-alternate"

synthetic_output = OUTPUT_ROOT / "synthetic" / "bootstrap" / "archive.ir.json"
write_archive_ir(archive, synthetic_output)
assert read_archive_ir(synthetic_output) == archive
assert json.loads(synthetic_output.read_text(encoding="utf-8"))["archive_version"] == "1.0"

print("Synthetic validation passed.")
print(f"Conversations: {summary.conversation_count}")
print(f"Graph nodes: {summary.node_count}")
print(f"Messages: {summary.message_count}")
print(f"Current-path messages: {summary.current_path_message_count}")
print(f"Warnings: {summary.warning_count}")
print(f"Archive IR: {synthetic_output}")

## Optional real-export run — disabled by default

Running this cell sends the selected export through a Google-hosted Colab runtime and writes a normalized copy to Google Drive. Only proceed if you deliberately accept that privacy boundary. Provide an exact existing Drive path; the notebook does not search, glob, or upload automatically. It prints only aggregate counts and an output path—not titles, messages, warning contexts, or the source filename.


In [ ]:
from datetime import UTC, datetime

RUN_REAL_EXPORT = False  # @param {type:"boolean"}
REAL_EXPORT_ZIP = ""  # @param {type:"string"}
PRIVACY_ACKNOWLEDGEMENT = ""  # @param {type:"string"}
REQUIRED_ACKNOWLEDGEMENT = "I UNDERSTAND THIS RUNS IN GOOGLE COLAB"

if not RUN_REAL_EXPORT:
    print("Real-export ingestion remains disabled.")
else:
    if PRIVACY_ACKNOWLEDGEMENT != REQUIRED_ACKNOWLEDGEMENT:
        raise RuntimeError("Exact privacy acknowledgment is required.")
    real_export_path = Path(REAL_EXPORT_ZIP).expanduser()
    if not real_export_path.is_file() or real_export_path.suffix.casefold() != ".zip":
        raise RuntimeError("REAL_EXPORT_ZIP must be an exact existing .zip file path.")
    run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    real_output_directory = OUTPUT_ROOT / "real" / run_id
    real_output_directory.mkdir(parents=True, exist_ok=False)
    try:
        real_archive = ingest_export_zip(
            real_export_path,
            limits=IngestLimits(),
            schema_mode=SchemaMode.TOLERANT,
        )
        real_output_path = write_archive_ir(real_archive, real_output_directory / "archive.ir.json")
        real_summary = summarize_archive(real_archive)
    except Exception as exception:
        raise RuntimeError(
            f"Real-export ingestion failed ({type(exception).__name__}); detailed exception "
            "text was suppressed to protect source data."
        ) from None
    print("Real-export ingestion completed.")
    print(f"Conversations: {real_summary.conversation_count}")
    print(f"Graph nodes: {real_summary.node_count}")
    print(f"Messages: {real_summary.message_count}")
    print(f"Current-path messages: {real_summary.current_path_message_count}")
    print(f"Warnings: {real_summary.warning_count}")
    print(f"Archive IR: {real_output_path}")